In [11]:
import dspy
import os
from dotenv import load_dotenv
from dspy.datasets import DataLoader
from datasets import load_dataset
import random

In [12]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [13]:
#lm = dspy.LM("ollama_chat/qwen3.5:0.8b", api_base="http://localhost:11434", api_key="")
lm = dspy.LM("openai/gpt-5.1", api_key=OPENAI_API_KEY)
dspy.configure(lm=lm)

In [21]:
classifier = dspy.Predict("text -> label")

In [14]:
# Load the IMDB dataset.
dataset = load_dataset("imdb")
CLASS_LABELS = dataset["train"].features["label"].names
kwargs = dict(fields=("text", "label"), input_keys=("text",), split="train", trust_remote_code=True)

In [25]:
# Load the first 1000 examples from the dataset, and assign a hint to each *training* example.
raw_data = [
    dspy.Example(x, label=CLASS_LABELS[x.label]).with_inputs("text")
    for x in DataLoader().from_huggingface(dataset_name="imdb", **kwargs)[:2000]
]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'imdb' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


In [26]:
random.Random(0).shuffle(raw_data)

In [27]:
unlabeled_trainset = [dspy.Example(text=x.text).with_inputs("text") for x in raw_data[:500]]

unlabeled_trainset[0]

Example({'text': "and a 30,000$ budget and this movie still looks like it was made for 50$. You can tell from the first frame to the last that he didn't care one bit about the movies continuity or plot, he was just happy to be making a zombie movie. <br /><br />What the end result shows is a lazy film maker who loves zombie movies. It could have been great if he just had of given a care. The end result is endless zoom ins on poorly done gore, and even more poorly produced metal plays over it.<br /><br />What happens when you combine high hopes, big dreams, a decent budget, hard work, and one idiot behind the camera."}) (input_keys={'text'})

In [32]:
devset = raw_data[600:700]
devset[2]

Example({'text': 'Any time a movie is so myopic in its desire to present a particular ending or viewpoint that it simply doesn\'t bother with an actual story, it\'s annoying. Those are the types of movies where the ending or viewpoint is conceived first, and the story simply tacked on. For this reason we often talk of the story "jumping through hoops" as it twists about, trying in vain to progress to the preordained ending in a logical fashion.<br /><br />The story in "Comet Over Broadway" doesn\'t just jump through hoops, it\'s a three ring circus. It\'s so ludicrous, so ill-conceived, so disingenuous that, if you are prone to speaking aloud to the screen, you will be carrying on quite a rant before it\'s through.<br /><br />The central theme of this screenplay cesspool is that of a woman choosing between family and profession. Since it\'s all so horribly muddled it will end up offensive to people of either opinion. So, in the end there\'s no point to the story, the theme becomes irre

In [33]:
metric = (lambda x, y, trace=None: x.label == y.label)
evaluate = dspy.Evaluate(devset=devset, metric=metric, display_progress=True, display_table=100)

In [34]:
evaluate(classifier)

  0%|          | 0/100 [00:00<?, ?it/s]

Average Metric: 0.00 / 100 (0.0%): 100%|██████████| 100/100 [00:14<00:00,  7.02it/s]

2026/03/09 17:00:15 INFO dspy.evaluate.evaluate: Average Metric: 0 / 100 (0.0%)


,text,example_label,pred_label,<lambda>
0,My wife and I both agree that this is one of the worst movies ever...,neg,negative,✔️ [False]
1,When Marlene Dietrich was labeled box office poison in 1938 one of...,neg,negative,✔️ [False]
2,Any time a movie is so myopic in its desire to present a particula...,neg,negative,✔️ [False]
3,"I can not believe such slanted, jingoistic material is getting pas...",neg,negative,✔️ [False]
4,"One scene demonstrates the mentality of ""Terminator Woman"" pretty ...",neg,negative,✔️ [False]
...,...,...,...,...
95,"What Is It? is a mish-mash of bizarre recurring motifs (snails, Sh...",neg,negative,✔️ [False]
96,Lorne Michaels once again proves that he has absolutely no busines...,neg,negative,✔️ [False]
97,"There was nothing of value in the original movie, this one was eve...",neg,negative,✔️ [False]
98,I can't believe they got the actors and actresses of that caliber ...,neg,positive,✔️ [False]


EvaluationResult(score=0.0, results=<list of 100 results>)